In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
def process_bed(s):
    bins_mc = pd.read_csv(f'{indir}/bed/bins/rna_cluster/{s}.chrom5k_CGN-Both.sparse.bed.gz', sep='\t', header=None)
    bins_mc.index = [f'{x}_{y}' for x,y in zip(bins_mc[0], bins_mc[1])]
    return bins_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_order), total=len(tmp_order)))

In [ ]:
[x.shape[0] for x in results]

In [ ]:
bins_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/windows/5kb/GRCh38.5kb.bed'
bins = pd.read_csv(bins_path, sep='\t', header=None)
bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
bins.shape

In [ ]:
bins.head()

In [ ]:
results_idx = [x.reindex(bins.index) for x in results]
len(results_idx)

In [ ]:
mc_idx = [x[4] / x[5] for x in results_idx]
bins_df = pd.DataFrame(index=bins.index)
for i,s in enumerate(tmp_order):
    bins_df[s] = mc_idx[i]
bins_df.dropna(inplace=True)
bins_df.shape

In [ ]:
bins_df.head()

In [ ]:
### note: corr df looks similar for chrom100k
corr_df = bins_df.corr()
corr_df

In [ ]:
X = bins_df.values.T
X_z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
bins_df_z = pd.DataFrame(X_z.T, index=bins_df.index, columns=bins_df.columns)
corr_z = bins_df_z.corr()
corr_z

In [ ]:
bins_df_z.head()

In [ ]:
bins_df_z.mean(axis=1)

In [ ]:
fig, axes = plt.subplots(1, figsize=(8,6))
ax = sns.heatmap(corr_df, vmin=0.5, vmax=1.0, cmap='coolwarm')
ax.set_title('Pearson correlation of 5kb methylation bins')

In [ ]:
fig, axes = plt.subplots(1, figsize=(8,6))
ax = sns.heatmap(corr_z, vmin=-1, vmax=1.0, cmap='coolwarm')
ax.set_title('Pearson correlation of 5kb methylation bins')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
corr_df.to_csv(f'{outdir}/mcg_celltype_5kb_bins_corr.csv', sep='\t', header=True, index=True)
corr_z.to_csv(f'{outdir}/mcg_celltype_5kb_bins_corr_z.csv', sep='\t', header=True, index=True)

### instead of the mean what if we just looked at the mean difference

In [ ]:
!date